<div style="background:#20beff;color:white;padding:20px 24px;border-radius:8px;margin-bottom:8px;">
  <h1 style="margin:0;font-size:1.8em;">🧬 ECABSD v2 — Kaggle GPU Training</h1>
  <p style="margin:6px 0 0;opacity:0.9;">Equivariant Cross-Attention Binding Site Detection</p>
</div>

**Note:** Ensure you have selected **GPU P100** or **GPU T4 x2** in the Kaggle Accelerator settings.

## Step 1 — Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected!\n"
        "Go to: Settings (right sidebar) → Accelerator → Select GPU P100 or T4 x2"
    )

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ GPU  : {gpu_name}")
print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2 — Install Dependencies

In [ ]:
!pip install -q torch_geometric biopython pyyaml pydssp tqdm scikit-learn matplotlib seaborn
print("✅ Dependencies installed")

## Step 3 — Setup Code & Data
Since your dataset only contains the data, we will download the code from GitHub and then merge your Kaggle dataset into it.

In [ ]:
import os
import shutil

WORKING_DIR = "/kaggle/working/ecabsd"

# 1. Clone the code repository
if not os.path.exists(WORKING_DIR):
    print("Cloning the code repository...")
    !git clone https://github.com/nayanees6607/ecabsd_temp.git {WORKING_DIR}
    print("✅ Code cloned successfully")

%cd {WORKING_DIR}

# 2. Automatically find your attached dataset
try:
    dataset_folder = os.listdir("/kaggle/input")[0]
    DATASET_PATH = f"/kaggle/input/{dataset_folder}"
    print(f"Found attached dataset at: {DATASET_PATH}")
    
    # 3. Copy the 'data' contents from your dataset into the working directory
    source_data = os.path.join(DATASET_PATH, "data")
    target_data = os.path.join(WORKING_DIR, "data")
    
    if os.path.exists(source_data):
        print("Copying data into the working directory...")
        shutil.copytree(source_data, target_data, dirs_exist_ok=True)
        print("✅ Data copied successfully")
    else:
        # If the dataset was uploaded without a top-level 'data' folder
        shutil.copytree(DATASET_PATH, target_data, dirs_exist_ok=True)
        print("✅ Data copied successfully")

except IndexError:
    print("⚠️ WARNING: No dataset found in /kaggle/input/. Make sure it is attached in the right sidebar.")

## Step 4 — Download BM5 PDBs (If not already processed)
If you uploaded `data/db5_processed` with the `.pt` files, you can skip this cell.

In [ ]:
import glob

PROCESSED_DIR = "data/db5_processed"

if len(glob.glob(f"{PROCESSED_DIR}/*.pt")) > 10:
    print("✅ Processed graphs found. Skipping DB5 prep.")
else:
    print("Processed graphs missing. Downloading BM5-clean and preparing...")
    !git clone https://github.com/haddocking/BM5-clean.git data/BM5-clean
    !python scripts/prepare_db5.py --db5-dir data/BM5-clean/HADDOCK-ready --output-dir data/db5_processed --threads 2
    print("✅ Dataset preparation complete.")

## Step 5 — Train the Model

In [ ]:
!python train.py

## Step 6 — Evaluate

In [ ]:
!python evaluate.py

## Step 7 — Save Checkpoints to Output
Zip the results so they are easy to download from Kaggle's output section.

In [ ]:
import shutil

print("Zipping outputs...")
shutil.make_archive("/kaggle/working/ecabsd_results", "zip", "/kaggle/working/ecabsd")
print("✅ Saved as ecabsd_results.zip in Kaggle Output tab")